In [1]:
import importlib
import os
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

import dtree_process_worker as _dtree_worker

# Force refresh in notebooks so newly added helpers are visible.
_dtree_worker = importlib.reload(_dtree_worker)

if hasattr(_dtree_worker, "build_class_folds_worker"):
    build_class_folds_worker = _dtree_worker.build_class_folds_worker
else:
    # Fallback keeps notebook runnable even if module is stale.
    def build_class_folds_worker(args):
        class_value, class_indices, k, seed = args
        indices = np.asarray(class_indices, dtype=np.int64).copy()
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)
        split_indices = np.array_split(indices, int(k))
#         return class_value, [chunk.astype(np.int64) for chunk in split_indices]


In [2]:
def entropy_from_labels(labels):
    labels_arr = np.asarray(labels, dtype=np.int64)
    if labels_arr.size == 0:
        return 0.0
    _, counts = np.unique(labels_arr, return_counts=True)
    probs = counts / counts.sum()
    return float(-np.sum(probs * np.log2(probs)))


def majority_label(labels):
    labels_arr = np.asarray(labels, dtype=np.int64)
    values, counts = np.unique(labels_arr, return_counts=True)
    return int(values[np.argmax(counts)])


def split_dataset_np(dataset, feature_index, feature_value):
    data = np.asarray(dataset)
    if data.size == 0:
        return data
    mask = data[:, feature_index] == feature_value
    filtered = data[mask]
    if filtered.size == 0:
        return np.empty((0, data.shape[1] - 1), dtype=data.dtype)
    return np.delete(filtered, feature_index, axis=1)


def _feature_gain_worker(args):
    dataset, feature_index, base_entropy = args
    data = np.asarray(dataset)
    values = np.unique(data[:, feature_index])
    total = float(data.shape[0])
    conditional_entropy = 0.0

    for value in values:
        subset = split_dataset_np(data, feature_index, value)
        if subset.shape[0] == 0:
            continue
        conditional_entropy += (subset.shape[0] / total) * entropy_from_labels(subset[:, -1])

    return feature_index, base_entropy - conditional_entropy


def best_feature_threaded(dataset, thread_workers=8):
    data = np.asarray(dataset)
    n_features = data.shape[1] - 1
    if n_features <= 1:
        return 0

    base_entropy = entropy_from_labels(data[:, -1])
    args = np.empty(n_features, dtype=object)
    for idx in range(n_features):
        args[idx] = (data, idx, base_entropy)

    workers = max(1, min(int(thread_workers), n_features))
    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = tuple(executor.map(_feature_gain_worker, args))

    return int(max(results, key=lambda item: item[1])[0])


def build_tree(dataset, feature_names, min_sample_size=1, max_depth=None, depth=0, thread_workers=8):
    data = np.asarray(dataset)
    names = np.asarray(feature_names, dtype=object)

    labels = data[:, -1]

    # stop condition1: when number of samples less then setted
    if np.unique(labels).size <= max(1, min_sample_size):
        return majority_label(labels)

    # stop condition2: run out of features
    if data.shape[1] == 1:
        return majority_label(labels)

    # stop condition3: reach setted depth
    if max_depth is not None and depth >= max_depth:
        return majority_label(labels)

    

    best_feature = best_feature_threaded(data, thread_workers=thread_workers)
    root_name = names[best_feature]
    tree = {root_name: {}}

    child_feature_names = np.delete(names, best_feature)
    unique_values = np.unique(data[:, best_feature])
    for value in unique_values:
        child_data = split_dataset_np(data, best_feature, value)
        if child_data.shape[0] == 0:
            tree[root_name][int(value)] = majority_label(labels)
        else:
            tree[root_name][int(value)] = build_tree(
                child_data,
                child_feature_names,
                min_sample_size = min_sample_size,
                max_depth=max_depth,
                depth=depth + 1,
                thread_workers=thread_workers,
            )

    return tree


def predict_one(tree, feature_names, sample, default_label):
    node = tree
    names = np.asarray(feature_names, dtype=object)
    x = np.asarray(sample)

    while isinstance(node, dict):
        root = next(iter(node))
        children = node[root]

        matched = np.where(names == root)[0]
        if matched.size == 0:
            return default_label

        idx = int(matched[0])
        value = int(x[idx])
        if value not in children:
            return default_label

        node = children[value]
        names = np.delete(names, idx)
        x = np.delete(x, idx)

    return int(node)


def predict_batch(tree, feature_names, dataset, default_label):
    data = np.asarray(dataset)
    x = data[:, :-1]
    preds = np.empty(x.shape[0], dtype=np.int64)
    for i in range(x.shape[0]):
        preds[i] = predict_one(tree, feature_names, x[i], default_label)
    return preds


def accuracy_np(y_true, y_pred):
    true_arr = np.asarray(y_true, dtype=np.int64)
    pred_arr = np.asarray(y_pred, dtype=np.int64)
    if true_arr.size == 0:
        return 0.0
    return float(np.mean(true_arr == pred_arr))


def post_prune_tree_reduced_error(tree, train_data, valid_data, feature_names):
    if not isinstance(tree, dict):
        return tree

    train_np = np.asarray(train_data)
    valid_np = np.asarray(valid_data)
    names = np.asarray(feature_names, dtype=object)

    if train_np.shape[0] == 0:
        return tree

    root = next(iter(tree))
    children = tree[root]
    root_idx = int(np.where(names == root)[0][0])
    child_names = np.delete(names, root_idx)

    pruned_children = {}
    for edge_val, child in children.items():
        child_train = split_dataset_np(train_np, root_idx, edge_val)
        child_valid = split_dataset_np(valid_np, root_idx, edge_val)
        pruned_children[edge_val] = post_prune_tree_reduced_error(child, child_train, child_valid, child_names)

    pruned_tree = {root: pruned_children}

    if valid_np.shape[0] == 0:
        return pruned_tree

    default = majority_label(train_np[:, -1])
    subtree_pred = predict_batch(pruned_tree, names, valid_np, default)
    subtree_acc = accuracy_np(valid_np[:, -1], subtree_pred)

    leaf_pred = np.full(valid_np.shape[0], default, dtype=np.int64)
    leaf_acc = accuracy_np(valid_np[:, -1], leaf_pred)

    if leaf_acc >= subtree_acc:
        return int(default)

    return pruned_tree

In [3]:
def count_leaf_nodes(tree):
    if not isinstance(tree, dict):
        return 1
    root = next(iter(tree))
    return int(sum(count_leaf_nodes(child) for child in tree[root].values()))


def tree_depth(tree):
    if not isinstance(tree, dict):
        return 1
    root = next(iter(tree))
    child_depths = np.fromiter(
        (tree_depth(child) for child in tree[root].values()),
        dtype=np.int64,
    )
    return int(1 + child_depths.max(initial=0))


def save_markdown_report(results_array, output_path):
    rows = np.asarray(results_array, dtype=object)
    lines = np.empty(rows.shape[0] + 8, dtype=object)
    lines[0] = "# Fold Results"
    lines[1] = ""
    lines[2] = "| Fold | Unpruned Acc | Pruned Acc | Leaves | Depth |"
    lines[3] = "|---:|---:|---:|---:|---:|"

    unpruned = np.empty(rows.shape[0], dtype=np.float64)
    pruned = np.empty(rows.shape[0], dtype=np.float64)

    for i, item in enumerate(rows):
        unpruned[i] = float(item["unpruned_accuracy"])
        pruned[i] = float(item["pruned_accuracy"])
        lines[i + 4] = (
            f"| {item['fold']} | {unpruned[i]:.4f} | {pruned[i]:.4f} | "
            f"{item['leaf_count']} | {item['depth']} |"
        )

    lines[-4] = ""
    lines[-3] = f"- Mean unpruned accuracy: {unpruned.mean():.4f}"
    lines[-2] = f"- Mean pruned accuracy: {pruned.mean():.4f}"
    lines[-1] = ""

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    output_file.write_text("\n".join(lines.tolist()), encoding="utf-8")
    print(f"Saved report: {output_file}")

# Datasets used in this notebook:
# 1) Letter Recognition
# 2) Adult
# 3) Mushroom
#
# Install once if needed:
# %pip install ucimlrepo

In [4]:
# UCI IDs: Letter=59, Adult=2, Mushroom=73
letter_df = fetch_ucirepo(id=59).data.original.copy()
adult_df = fetch_ucirepo(id=2).data.original.copy()
mushroom_df = fetch_ucirepo(id=73).data.original.copy()

print(letter_df.shape, adult_df.shape, mushroom_df.shape)
print("Letter columns:", letter_df.columns.to_numpy(dtype=object))


(20000, 17) (48842, 15) (8124, 23)
Letter columns: ['lettr' 'x-box' 'y-box' 'width' 'high' 'onpix' 'x-bar' 'y-bar' 'x2bar'
 'y2bar' 'xybar' 'x2ybr' 'xy2br' 'x-ege' 'xegvy' 'y-ege' 'yegvx']


In [5]:
def encode_dataframe_to_int_numpy(df, target_col, random_state=42):
    cols = df.columns.to_numpy(dtype=object)
    working_df = df.copy()
    rng = np.random.default_rng(random_state)

    # Drop feature columns that are entirely missing.
    drop_cols = []
    for col_name in cols:
        if col_name == target_col:
            continue
        if working_df[col_name].isna().all():
            drop_cols.append(col_name)
    if drop_cols:
        working_df = working_df.drop(columns=drop_cols)
        cols = working_df.columns.to_numpy(dtype=object)

    # Impute missing values only in features by sampling from each feature's
    # empirical distribution in observed (non-missing) values.
    for col_name in cols:
        if col_name == target_col:
            continue
        series = working_df[col_name]
        missing_mask = series.isna()
        if not missing_mask.any():
            continue

        observed = series[~missing_mask]
        if observed.shape[0] == 0:
            # Entirely-missing columns are dropped above.
            continue

        value_probs = observed.value_counts(dropna=True, normalize=True)
        sampled_values = rng.choice(
            value_probs.index.to_numpy(dtype=object),
            size=int(missing_mask.sum()),
            p=value_probs.to_numpy(dtype=np.float64),
        )
        working_df.loc[missing_mask, col_name] = sampled_values

    encoded = np.empty((working_df.shape[0], working_df.shape[1]), dtype=np.int64)
    for col_idx, col_name in enumerate(cols):
        series = working_df[col_name]
        if pd.api.types.is_numeric_dtype(series):
            encoded[:, col_idx] = pd.to_numeric(series, errors="coerce").fillna(0).to_numpy(dtype=np.int64)
        else:
            codes, _ = pd.factorize(series.astype(str), sort=True)
            encoded[:, col_idx] = codes.astype(np.int64)

    target_idx = int(np.where(cols == target_col)[0][0])
    feature_indices = np.delete(np.arange(cols.shape[0], dtype=np.int64), target_idx)
    feature_names = cols[feature_indices]

    dataset = np.concatenate(
        [encoded[:, feature_indices], encoded[:, target_idx:target_idx + 1]],
        axis=1,
    )
    return dataset, feature_names


def create_kfold_datasets_multiprocess(
    dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1.0,
    random_state=42,
    process_workers=None,
):
    data = np.asarray(dataset, dtype=np.int64)
    y = data[:, -1]

    rng = np.random.default_rng(random_state)
    class_values = np.unique(y)

    sampled_indices = np.empty(0, dtype=np.int64)
    for class_value in class_values:
        class_idx = np.where(y == class_value)[0]
        class_idx = rng.permutation(class_idx)
        if sample_size_ratio >= 1.0:
            take_count = class_idx.shape[0]
        else:
            take_count = max(1, int(class_idx.shape[0] * sample_size_ratio))
        sampled_indices = np.concatenate([sampled_indices, class_idx[:take_count]])

    sampled_indices = rng.permutation(sampled_indices)
    sampled_data = data[sampled_indices]
    y_sampled = sampled_data[:, -1]

    # 1) Build one fixed stratified test split.
    fixed_test_indices = np.empty(0, dtype=np.int64)
    trainval_indices = np.empty(0, dtype=np.int64)
    sampled_all_idx = np.arange(sampled_data.shape[0], dtype=np.int64)

    for class_value in np.unique(y_sampled):
        class_idx = sampled_all_idx[y_sampled == class_value]
        class_idx = rng.permutation(class_idx)
        test_count = int(class_idx.shape[0] * test_ratio)
        test_count = max(1, test_count)
        test_count = min(test_count, class_idx.shape[0] - 1)
        fixed_test_indices = np.concatenate([fixed_test_indices, class_idx[:test_count]])
        trainval_indices = np.concatenate([trainval_indices, class_idx[test_count:]])

    fixed_test_indices = rng.permutation(fixed_test_indices)
    trainval_indices = rng.permutation(trainval_indices)
    fixed_test_data = sampled_data[fixed_test_indices]
    trainval_data = sampled_data[trainval_indices]
    y_trainval = trainval_data[:, -1]

    workers = process_workers
    if workers is None:
        workers = max(1, min(k, (os.cpu_count() or 1) - 1))

    task_args = np.empty(class_values.shape[0], dtype=object)
    for i, class_value in enumerate(class_values):
        class_local_idx = np.where(y_trainval == class_value)[0]
        task_args[i] = (int(class_value), class_local_idx, int(k), int(random_state + 1000 + i))

    with ProcessPoolExecutor(max_workers=workers) as executor:
        class_chunks = tuple(executor.map(build_class_folds_worker, task_args))

    fold_validation_indices = np.empty(k, dtype=object)
    for i in range(k):
        fold_validation_indices[i] = np.empty(0, dtype=np.int64)

    for _, chunks in class_chunks:
        for fold_idx in range(k):
            fold_validation_indices[fold_idx] = np.concatenate([fold_validation_indices[fold_idx], chunks[fold_idx]])

    for fold_idx in range(k):
        fold_validation_indices[fold_idx] = rng.permutation(fold_validation_indices[fold_idx])

    all_indices = np.arange(trainval_data.shape[0], dtype=np.int64)
    fold_records = np.empty(k, dtype=object)

    for fold_idx in range(k):
        val_idx = fold_validation_indices[fold_idx]
        train_mask = np.ones(trainval_data.shape[0], dtype=bool)
        train_mask[val_idx] = False
        train_idx = all_indices[train_mask]

        fold_records[fold_idx] = {
            "fold": fold_idx + 1,
            "train": trainval_data[train_idx],
            "validation": trainval_data[val_idx],
            "test": fixed_test_data,
        }

    return fold_records


letter_dataset, letter_dataset_feature_names = encode_dataframe_to_int_numpy(letter_df, target_col="lettr")
adult_dataset, adult_dataset_feature_names = encode_dataframe_to_int_numpy(adult_df, target_col="income")
mushroom_dataset, mushroom_dataset_feature_names = encode_dataframe_to_int_numpy(mushroom_df, target_col="poisonous")

print(letter_dataset_feature_names)
print(adult_dataset_feature_names)
print(mushroom_dataset_feature_names)

letter_fold_datasets = create_kfold_datasets_multiprocess(
    letter_dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

adult_fold_datasets = create_kfold_datasets_multiprocess(
    adult_dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

mushroom_fold_datasets = create_kfold_datasets_multiprocess(
    mushroom_dataset,
    k=10,
    test_ratio=0.1,
    sample_size_ratio=1,
    random_state=42,
)

dataset_names = ["letter", "adult", "mushroom"]
original_datasets = [letter_dataset, adult_dataset, mushroom_dataset]
folded_datasets = [letter_fold_datasets, adult_fold_datasets, mushroom_fold_datasets]


for dataset_name, original_dataset, folded_dataset in zip(dataset_names, original_datasets, folded_datasets):
    target_categories = np.unique(original_dataset[:, -1])
    missing_count = int(np.isnan(original_dataset.astype(np.float64)).sum())
    print(f"dataset: {dataset_name}")
    print("dataset shape:", original_dataset.shape)
    print("target categories:", target_categories.tolist())
    print("missing values after preprocessing:", missing_count)
    print("has missing values:", missing_count > 0)
    print("dataset preview (first 5 rows):")
    print(original_dataset[:5])
    print("validation size:", folded_dataset[0]["validation"].shape[0])
    for fold in folded_dataset:
        print(
            f"fold {fold['fold']}: train={fold['train'].shape[0]}, "
            f"test={fold['test'].shape[0]}, validation={fold['validation'].shape[0]}"
        )
    print("\n")



['x-box' 'y-box' 'width' 'high' 'onpix' 'x-bar' 'y-bar' 'x2bar' 'y2bar'
 'xybar' 'x2ybr' 'xy2br' 'x-ege' 'xegvy' 'y-ege' 'yegvx']
['age' 'workclass' 'fnlwgt' 'education' 'education-num' 'marital-status'
 'occupation' 'relationship' 'race' 'sex' 'capital-gain' 'capital-loss'
 'hours-per-week' 'native-country']
['cap-shape' 'cap-surface' 'cap-color' 'bruises' 'odor' 'gill-attachment'
 'gill-spacing' 'gill-size' 'gill-color' 'stalk-shape' 'stalk-root'
 'stalk-surface-above-ring' 'stalk-surface-below-ring'
 'stalk-color-above-ring' 'stalk-color-below-ring' 'veil-type'
 'veil-color' 'ring-number' 'ring-type' 'spore-print-color' 'population'
 'habitat']
dataset: letter
dataset shape: (20000, 17)
target categories: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
missing values after preprocessing: 0
has missing values: False
dataset preview (first 5 rows):
[[ 2  8  3  5  1  8 13  0  6  6 10  8  0  8  0  8 19]
 [ 5 12  3  7  2 10  5  5  4 13  3  9

# Train and evaluate all cases 

In [6]:
import sys
import time

def prepruning_grid_search_fold(
    fold_record,
    feature_names,
    min_sample_sizes,
    max_depths,
    thread_workers=8,
    show_process=True,
):
    t0 = time.perf_counter()
    train_data = np.asarray(fold_record["train"], dtype=np.int64)
    test_data = np.asarray(fold_record["test"], dtype=np.int64)
    validation_data = np.asarray(fold_record["validation"], dtype=np.int64)

    fold_id = fold_record.get("fold", "?")
    grid_total = len(list(min_sample_sizes)) * len(list(max_depths))
    if show_process:
        print(
            f"[fold {fold_id}] start | train={train_data.shape[0]} test={test_data.shape[0]} "
            f"val={validation_data.shape[0]} | grid={grid_total} configs",
            flush=True,
        )

    default_label = majority_label(train_data[:, -1])
    names = np.asarray(feature_names, dtype=object)

    best = {
        "min_sample_size": None,
        "max_depth": None,
        "selection_validation_accuracy": -1.0,
        "tree": None,
    }

    for k, (mss, mxd) in enumerate(
        ((int(mss), mxd) for mss in min_sample_sizes for mxd in max_depths),
        start=1,
    ):
        tree = build_tree(
            train_data,
            names,
            min_sample_size=mss,
            max_depth=mxd,
            thread_workers=thread_workers,
        )
        val_pred = predict_batch(tree, feature_names, validation_data, default_label)
        val_acc = accuracy_np(validation_data[:, -1], val_pred)
        leaves = count_leaf_nodes(tree)
        if val_acc > best["selection_validation_accuracy"] or (
            val_acc == best["selection_validation_accuracy"]
            and best["tree"] is not None
            and leaves < count_leaf_nodes(best["tree"])
        ):
            best.update(
                min_sample_size=mss,
                max_depth=mxd,
                selection_validation_accuracy=float(val_acc),
                tree=tree,
            )
        # Optional: sparse progress (e.g. every 10% of grid)
        if show_process and grid_total >= 50 and (k % max(1, grid_total // 10) == 0):
            print(f"[fold {fold_id}] grid {k}/{grid_total} ... best val(sel)={best['selection_validation_accuracy']:.4f}", flush=True)

    tree = best["tree"]
    test_pred = predict_batch(tree, feature_names, test_data, default_label)
    test_acc = accuracy_np(test_data[:, -1], test_pred)
    elapsed = time.perf_counter() - t0

    if show_process:
        print(
            f"[fold {fold_id}] done in {elapsed:.1f}s | "
            f"best mss={best['min_sample_size']} max_depth={best['max_depth']} | "
            f"val(sel)={best['selection_validation_accuracy']:.4f} | "
            f"test(report)={float(test_acc):.4f} | "
            f"leaves={count_leaf_nodes(tree)} depth={tree_depth(tree)}",
            flush=True,
        )

    return {
        "fold": fold_record,
        "best_min_sample_size": best["min_sample_size"],
        "best_max_depth": best["max_depth"],
        "validation_accuracy_used_to_select_hyperparameters": best["selection_validation_accuracy"],
        "test_accuracy": float(test_acc),
        "tree": tree,
        "leaf_count": count_leaf_nodes(tree),
        "depth": tree_depth(tree),
    }


def run_prepruning_all_folds(
    fold_datasets,
    feature_names,
    min_sample_sizes,
    max_depths,
    thread_workers=8,
):
    results = []
    for i in range(len(fold_datasets)):
        results.append(
            prepruning_grid_search_fold(
                fold_datasets[i],
                np.asarray(feature_names, dtype=object),
                min_sample_sizes=min_sample_sizes,
                max_depths=max_depths,
                thread_workers=thread_workers,
            )
        )
    test_accs = np.array([r["test_accuracy"] for r in results], dtype=np.float64)
    val_sel = np.array(
        [r["validation_accuracy_used_to_select_hyperparameters"] for r in results],
        dtype=np.float64,
    )
    return {
        "per_fold": results,
        "mean_test_accuracy": float(test_accs.mean()),
        "std_test_accuracy": float(test_accs.std(ddof=1)) if test_accs.size > 1 else 0.0,
        "mean_validation_accuracy_for_selection": float(val_sel.mean()),
        "test_accuracies": test_accs,
    }

### Stump vs unpruned vs post-pruned

- **Stump:** `max_depth=1`
- **Unpruned:** `min_sample_size=1`, `max_depth=None`
- **Pruned:** pre-tuning, grid search on min_sample_size and max_depth

In [7]:
# helpers for stump vs unpruned vs pre-pruned (no post-pruning)

import numpy as np
import pandas as pd

def _fit_and_score_case(
    train_data,
    eval_data,  # validation set
    feature_names,
    min_sample_size,
    max_depth,
    thread_workers=8,
):
    default_label = majority_label(train_data[:, -1])
    tree = build_tree(
        train_data,
        np.asarray(feature_names, dtype=object),
        min_sample_size=int(min_sample_size),
        max_depth=max_depth,
        thread_workers=thread_workers,
    )
    eval_pred = predict_batch(tree, feature_names, eval_data, default_label)
    eval_acc = accuracy_np(eval_data[:, -1], eval_pred)
    return {
        "tree": tree,
        "validation_accuracy": float(eval_acc),
        "leaf_count": int(count_leaf_nodes(tree)),
        "depth": int(tree_depth(tree)),
    }


def evaluate_three_models_one_fold(
    fold_record,
    feature_names,
    min_sample_sizes,
    thread_workers=8,
):
    train_data = np.asarray(fold_record["train"], dtype=np.int64)
    test_data = np.asarray(fold_record["test"], dtype=np.int64)          # final unbiased metric
    val_data = np.asarray(fold_record["validation"], dtype=np.int64)     # used for pre-pruned selection
    fold_id = int(fold_record["fold"])

    # 1) Stump: max_depth=1, report on validation
    stump = _fit_and_score_case(
        train_data=train_data,
        eval_data=val_data,
        feature_names=feature_names,
        min_sample_size=1,
        max_depth=1,
        thread_workers=thread_workers,
    )

    # 2) Unpruned: min_sample_size=1, max_depth=None, report on validation
    unpruned = _fit_and_score_case(
        train_data=train_data,
        eval_data=val_data,
        feature_names=feature_names,
        min_sample_size=1,
        max_depth=None,
        thread_workers=thread_workers,
    )

    # unbiased final test metrics for fixed-model baselines
    default_label = majority_label(train_data[:, -1])
    stump_test_pred = predict_batch(stump["tree"], feature_names, test_data, default_label)
    stump_test_acc = float(accuracy_np(test_data[:, -1], stump_test_pred))
    unpruned_test_pred = predict_batch(unpruned["tree"], feature_names, test_data, default_label)
    unpruned_test_acc = float(accuracy_np(test_data[:, -1], unpruned_test_pred))

    # Dynamic max_depth search space based on unpruned tree depth
    unpruned_depth = int(unpruned["depth"])
    dynamic_max_depths = list(range(0, unpruned_depth + 1))
    print("max_depths:", dynamic_max_depths)

    # 3) Pre-pruned: grid search on VALIDATION set, final report on TEST set
    best = {
        "min_sample_size": None,
        "max_depth": None,
        "selection_validation_accuracy": -1.0,
        "tree": None,
        "leaf_count": None,
        "depth": None,
    }

    # count total combinations
    num_grid_configs = len(list(min_sample_sizes)) * len(dynamic_max_depths)
    grid_checked = 0

    for mss in min_sample_sizes:
        for md in dynamic_max_depths:
            grid_checked += 1

            tree = build_tree(
                train_data,
                np.asarray(feature_names, dtype=object),
                min_sample_size=int(mss),
                max_depth=md,
                thread_workers=thread_workers,
            )

            # selection metric on validation set
            val_pred = predict_batch(tree, feature_names, val_data, default_label)
            val_acc = float(accuracy_np(val_data[:, -1], val_pred))
            leaves = int(count_leaf_nodes(tree))
            depth = int(tree_depth(tree))
            depth_aligned = depth - 1

            # print most recently calculated parameters + result
            print(
                f"[fold {fold_id}] {grid_checked}/{num_grid_configs} "
                f"| recent mss={int(mss)} max_depth={md} "
                f"| val_acc={val_acc:.4f} leaves={leaves} depth={depth_aligned}",
                flush=True
            )

            # tie-break: higher selection metric, then fewer leaves, then lower depth
            is_better = (
                (val_acc > best["selection_validation_accuracy"]) or
                (
                    val_acc == best["selection_validation_accuracy"]
                    and best["leaf_count"] is not None
                    and leaves < best["leaf_count"]
                ) or
                (
                    val_acc == best["selection_validation_accuracy"]
                    and leaves == best["leaf_count"]
                    and best["depth"] is not None
                    and depth < best["depth"]
                )
            )

            if is_better:
                best.update(
                    min_sample_size=int(mss),
                    max_depth=md,
                    selection_validation_accuracy=val_acc,
                    tree=tree,
                    leaf_count=leaves,
                    depth=depth,
                )
                print(
                    f"  -> NEW BEST: mss={best['min_sample_size']} max_depth={best['max_depth']} "
                    f"| val(sel)={best['selection_validation_accuracy']:.4f}",
                    flush=True
                )

    pre_tree = best["tree"]
    pre_test_pred = predict_batch(pre_tree, feature_names, test_data, default_label)
    pre_test_acc = float(accuracy_np(test_data[:, -1], pre_test_pred))

    return [
        {
            "fold": fold_id,
            "model": "stump",
            "validation_accuracy": stump["validation_accuracy"],
            "leaf_count": stump["leaf_count"],
            "depth": stump["depth"],
            "best_min_sample_size": 1,
            "best_max_depth": 1,
            "selection_validation_accuracy": np.nan,
            "test_accuracy": stump_test_acc,
        },
        {
            "fold": fold_id,
            "model": "unpruned",
            "validation_accuracy": unpruned["validation_accuracy"],
            "leaf_count": unpruned["leaf_count"],
            "depth": unpruned["depth"],
            "best_min_sample_size": 1,
            "best_max_depth": None,
            "selection_validation_accuracy": np.nan,
            "test_accuracy": unpruned_test_acc,
        },
        {
            "fold": fold_id,
            "model": "pre_pruned",
            "validation_accuracy": best["selection_validation_accuracy"],
            "leaf_count": best["leaf_count"],
            "depth": best["depth"],
            "best_min_sample_size": best["min_sample_size"],
            "best_max_depth": best["max_depth"],
            "selection_validation_accuracy": best["selection_validation_accuracy"],
            "test_accuracy": pre_test_acc,
        },
    ]

In [ ]:
# run all datasets and summarize

from IPython.display import display

# You can widen/narrow this grid depending on runtime.
# Keep None if you want "no depth cap" to be a candidate in pre-pruning.
min_sample_sizes = list(range(1, 10, 1))

configs = [
    ("letter", letter_fold_datasets, letter_dataset_feature_names),
    ("adult", adult_fold_datasets, adult_dataset_feature_names),
    ("mushroom", mushroom_fold_datasets, mushroom_dataset_feature_names),
]

all_rows = []

for dataset_name, folds, feature_names in configs:
    print(f"\n=== {dataset_name} ===")
    for fold_record in folds:
        fold_rows = evaluate_three_models_one_fold(
            fold_record=fold_record,
            feature_names=feature_names,
            min_sample_sizes=min_sample_sizes,
            thread_workers=8,
        )
        for r in fold_rows:
            r["dataset"] = dataset_name
            all_rows.append(r)

results_df = pd.DataFrame(all_rows)
results_df["depth_aligned_to_max_depth"] = results_df["depth"] - 1


# Per-fold detailed results
display(results_df.sort_values(["dataset", "fold", "model"]).reset_index(drop=True))

# Mean/std comparison table
summary_df = (
    results_df
    .groupby(["dataset", "model"], as_index=False)
    .agg(
        mean_test_accuracy=("test_accuracy", "mean"),
        std_test_accuracy=("test_accuracy", "std"),
        mean_leaf_count=("leaf_count", "mean"),
        mean_depth=("depth_aligned_to_max_depth", "mean"),
    )
    .sort_values(["dataset", "mean_test_accuracy"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n=== Summary (mean ± std test accuracy) ===")
display(summary_df)

# Optional: show pre-pruned chosen hyperparameters frequency
pre_choices = (
    results_df[results_df["model"] == "pre_pruned"]
    .groupby(["dataset", "best_min_sample_size", "best_max_depth"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["dataset", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

print("\n=== Pre-pruned hyperparameter choices ===")
display(pre_choices)


=== letter ===
max_depths: [0, 1, 2, 3, 4, 5, 6, 7, 8]
[fold 1] 1/81 | recent mss=1 max_depth=0 | val_acc=0.0408 leaves=1 depth=0
  -> NEW BEST: mss=1 max_depth=0 | val(sel)=0.0408
[fold 1] 2/81 | recent mss=1 max_depth=1 | val_acc=0.1703 leaves=16 depth=1
  -> NEW BEST: mss=1 max_depth=1 | val(sel)=0.1703
[fold 1] 3/81 | recent mss=1 max_depth=2 | val_acc=0.3975 leaves=183 depth=2
  -> NEW BEST: mss=1 max_depth=2 | val(sel)=0.3975
[fold 1] 4/81 | recent mss=1 max_depth=3 | val_acc=0.6229 leaves=1157 depth=3
  -> NEW BEST: mss=1 max_depth=3 | val(sel)=0.6229
[fold 1] 5/81 | recent mss=1 max_depth=4 | val_acc=0.7437 leaves=3567 depth=4
  -> NEW BEST: mss=1 max_depth=4 | val(sel)=0.7437
[fold 1] 6/81 | recent mss=1 max_depth=5 | val_acc=0.7409 leaves=5587 depth=5
[fold 1] 7/81 | recent mss=1 max_depth=6 | val_acc=0.7398 leaves=5903 depth=6
[fold 1] 8/81 | recent mss=1 max_depth=7 | val_acc=0.7404 leaves=5912 depth=7
[fold 1] 9/81 | recent mss=1 max_depth=8 | val_acc=0.7404 leaves=5912 d

In [ ]:
display(pre_choices)